# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Frederic7/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


**Unit of analysis.** One row = one pseudonymized content page, uniquely identified by
`content_id`. Each page belongs to one `client_id` pseudonym. The starter slice contains
30,000 pages across 32 clients. Verified below: 0 duplicate `content_id` values in the
30,000 rows.

**Time window.** All metrics on a row are aggregated over a single trailing 90-day window
ending at the CSV's export snapshot date (the exact date is not encoded in the file — see
Section 4, Data limits). Columns are grouped into three sub-windows:

- `*_90d` (e.g. `impressions_90d`, `clicks_90d`): totals over the full 90-day window.
- `*_last_30d`: days 0–30 back from the snapshot (the most recent 30 days).
- `*_prev_30d`: days 31–60 back (the 30-day period before `*_last_30d`).

`trend_pct` and `trend_direction` compare `*_last_30d` to `*_prev_30d` impressions. Days
61–90 back contribute to `*_90d` totals but have no dedicated comparison column.


> Measurement note: `impressions_*` and `days_with_impressions` come from Google Search
> Console (organic search only). `sessions_*`, `pageviews_*`, `users_*`, and
> `days_with_sessions` come from Google Analytics 4 (all traffic channels). So a page can
> legitimately have sessions on a day with no GSC impressions (direct, referral, social, or
> AI-referred traffic). See Section 4 for cross-system limits.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 1 code: grain probe, row counts, window-column inventory
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np

# --- Portable repo-root walk-up (works from repo root OR work/notebooks/) ---
NB_PATH = Path(os.path.abspath('')).resolve()
REPO_ROOT = NB_PATH
for _ in range(8):
    if (REPO_ROOT / 'data' / 'raw').is_dir() and (REPO_ROOT / 'scripts' / 'ml_utils.py').is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError(
        'Could not locate repo root. Expected a folder with data/raw/ and '
        f'scripts/ml_utils.py. Search started from: {NB_PATH}'
    )
SCRIPTS_DIR = str(REPO_ROOT / 'scripts')
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
RAW_PATH = REPO_ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
assert RAW_PATH.is_file(), f'Missing starter CSV: {RAW_PATH}'
print(f'Repo root: {REPO_ROOT}')
print(f'Starter CSV: {RAW_PATH.stat().st_size / 1024 / 1024:.1f} MB')

# --- Load ---
df = pd.read_csv(RAW_PATH)

# --- 1. Grain probe: content_id should be unique (0 dupes means 1 row = 1 page) ---
total_rows = len(df)
unique_content_ids = df['content_id'].nunique()
dup_check = (
    df.groupby('content_id', dropna=False)
      .size()
      .rename('rows_per_id')
      .reset_index()
      .query('rows_per_id > 1')
)
print('\n=== GRAIN ===')
print(f'Total rows            : {total_rows:,}')
print(f'Unique content_id     : {unique_content_ids:,}')
print(f'Duplicate content_ids : {len(dup_check)}  (expect 0)')
print(f'Unique client_id      : {df["client_id"].nunique()}  (expect 32)')

# --- 2. Window column inventory ---
cols = list(df.columns)
cols_90d = [c for c in cols if c.endswith('_90d')]
cols_last30 = [c for c in cols if c.endswith('_last_30d')]
cols_prev30 = [c for c in cols if c.endswith('_prev_30d')]
print('\n=== WINDOW COLUMNS ===')
print(f'*_90d       ({len(cols_90d)}):  {cols_90d}')
print(f'*_last_30d  ({len(cols_last30)}):  {cols_last30}')
print(f'*_prev_30d  ({len(cols_prev30)}):  {cols_prev30}')

# --- 3. Range checks that confirm the 90-day window story ---
print('\n=== WINDOW SANITY (expect min/max bounded by 90-day semantics) ===')
print(f'content_age_days         min={df["content_age_days"].min():.0f}  median={df["content_age_days"].median():.0f}  max={df["content_age_days"].max():.0f}  (all >= 90 in this slice)')
print(f'days_with_impressions    min={df["days_with_impressions"].min():.0f}  median={df["days_with_impressions"].median():.0f}  max={df["days_with_impressions"].max():.0f}  (0-90 range expected)')
print(f'days_with_sessions       min={df["days_with_sessions"].min():.0f}  median={df["days_with_sessions"].median():.0f}  max={df["days_with_sessions"].max():.0f}  (0-90 range expected)')
print(f'impressions_90d          min={df["impressions_90d"].min():.0f}  (> 0 expected; prep filter requires it)')

# --- 4. Cross-system gap: days_with_sessions can exceed days_with_impressions ---
gap = df['days_with_sessions'] - df['days_with_impressions']
n_gap_positive = int((gap > 0).sum())
n_gap_zero = int((gap == 0).sum())
n_gap_negative = int((gap < 0).sum())
print('\n=== CROSS-SYSTEM: days_with_sessions vs days_with_impressions ===')
print(f'Rows with sessions_days > impressions_days : {n_gap_positive:,}  ({n_gap_positive/total_rows*100:.1f}%)')
print(f'Rows with sessions_days = impressions_days : {n_gap_zero:,}  ({n_gap_zero/total_rows*100:.1f}%)')
print(f'Rows with sessions_days < impressions_days : {n_gap_negative:,}  ({n_gap_negative/total_rows*100:.1f}%)')
if n_gap_positive:
    print(f'  Positive gap (extra session-days): min={gap[gap>0].min()}, median={gap[gap>0].median():.1f}, max={gap[gap>0].max()}')
if n_gap_negative:
    print(f'  Negative gap (extra impression-days): min={gap[gap<0].min()} (most negative), median={gap[gap<0].median():.1f}')

Repo root: /Users/frederic/Desktop/git-repo/flyrank-internship-ml
Starter CSV: 6.4 MB

=== GRAIN ===
Total rows            : 30,000
Unique content_id     : 30,000
Duplicate content_ids : 0  (expect 0)
Unique client_id      : 32  (expect 32)

=== WINDOW COLUMNS ===
*_90d       (8):  ['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']
*_last_30d  (3):  ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']
*_prev_30d  (3):  ['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

=== WINDOW SANITY (expect min/max bounded by 90-day semantics) ===
content_age_days         min=90  median=236  max=564  (all >= 90 in this slice)
days_with_impressions    min=1  median=81  max=88  (0-90 range expected)
days_with_sessions       min=1  median=6  max=90  (0-90 range expected)
impressions_90d          min=1  (> 0 expected; prep filter requires it)

=== CROSS-SYSTEM: days_with_sessions vs d

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


Every field the pipeline touches (44 raw CSV columns + 8 columns added by `scripts/01_prepare_features.py`), sorted into four buckets. Excluded fields each carry a one-line why.

### Context (grouping / splitting / reading only; never model-learned)

- `content_id` — pseudonymous page key; unique row grain; for joins and grouping, never a feature.
- `client_id` — pseudonymous client key; 32 values in the starter slice. Used for client-holdout train/test splits.

### Label / proxy (y train/test splits only).

Only used to create `y` or its direct inputs. **Never a feature.

- `trend_direction` — Label precursor; `trend_pct` — raw label input to `trend_direction`.  `_declining_label` — **The target.** 1 = `trend_direction == "down"`. **impressions_last_30d** — Direct numerator input `trend_pct` formula.
- `impressions_prev_30d` — Direct denominator input to `trend_pct` formula.
- `clicks_last_30d`, `sessions_last_30d` — Same 30-day window as the label's numerator. Contemporaneous with the outcome being labeled — leakage-adjacent; excluded per the label-trap rule.
- `clicks_prev_30d`, `sessions_prev_30d` — Same 30-day window as the label's denominator. Mirror same logic as `impressions_prev_30d; excluded from features per window-alignment rule.

### Features (knowable at prediction moment; exactly the contents of `MODEL_NUMERIC_FEATURES` + `MODEL_CATEGORICAL_FEATURES` in ml_utils.py)

**Numeric features (19 columns):**
- Keyword context: `search_volume`, `competition`, `cpc`
- Content properties: `word_count`, `char_count`, `content_age_days`, `days_since_last_update`
- Derived 90-day (logged totals (logged):  `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d`
- 90-day activity counts: `days_with_impressions`, `days_with_sessions`
- Derived rates (×100 percentages): `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`
- Position: `avg_position` (0 = no data, handled downstream)

**Categorical features (9 columns):**
- Keyword context: `competition_level`, `content_type`, `main_intent`
- Property tiers: `age_tier`, `freshness_tier`, `word_count_tier`
- Performance tiers: `impression_tier`, `position_tier`

### Excluded (why)

Each field is excluded from features AND labels (because it leaks, encodes production context, or is a duplicate-derived signal already captured above:

- `provider_used` — Production metadata about which LLM vendor generated the article; a product decision, not a user signal. Data dictionary marks it "Not a model feature."
- `model_used` — LLM model name; same reason as `provider_used`.
- `char_count_tier` — Duplicate signal to `word_count_tier` (correlated near 1.0); drop one to avoid multicollinearity. Model categorical features exclude it.
- `impressions_90d`, `clicks_90d`, `sessions_90d`, `ai_sessions_90d`, `engaged_sessions_90d`, `pageviews_90d`, `users_90d`, `scroll_events_90d` — Raw 90-day totals are replaced by their `log_*` transforms in the model (traffic;logged variants go into features; `engaged_sessions_90d`, `pageviews_90d`, `users_90d`, `scroll_events_90d` excluded because derived rates `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` already capture the ratio — raw totals double-count.
- `age_tier_order` — Numeric ordinal duplicate of the categorical `age_tier`; the model uses the categorical.
- `has_clicks`, `has_ai_sessions` — Binary flags of already captured by the log-transforms (log1p(0)=0 distinguishes zero from non-zero implicitly; excluded to avoid duplicate signal.
- `measurable_opportunity` — Downstream filter flag, not a predictive signal. Excluded so the model does not learn a rule threshold by content-type-specifically:  `users_90d` — Used by rule filter; excluded to avoid duplicate signal with `sessions_90d` and rate columns.



In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 2 code: verify field buckets are complete, no overlaps, matches pipeline
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np

# --- Portable repo-root walk-up ---
NB_PATH = Path(os.path.abspath('')).resolve()
REPO_ROOT = NB_PATH
for _ in range(8):
    if (REPO_ROOT / 'data' / 'raw').is_dir() and (REPO_ROOT / 'scripts' / 'ml_utils.py').is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError(f'Repo root not found from {NB_PATH}')
SCRIPTS_DIR = str(REPO_ROOT / 'scripts')
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
RAW_PATH = REPO_ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
assert RAW_PATH.is_file(), f'Missing CSV: {RAW_PATH}'

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

# --- Load raw CSV + add 8 prep-script columns ---
df = pd.read_csv(RAW_PATH)
prep_cols_from_script = [
    # mimic the prep script's added columns for bucket check
]
# Build the full 52-column universe
# 44 raw + 8 prep-added
prep_added = [
    'is_declining_label',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'has_clicks', 'has_ai_sessions', 'measurable_opportunity',
]

# --- Define our four buckets (match the markdown above ---
CONTEXT = ['content_id', 'client_id']
LABEL_PROXY = [
    'trend_direction', 'trend_pct', 'is_declining_label',
    'impressions_last_30d', 'impressions_prev_30d',
    'clicks_last_30d', 'sessions_last_30d',
    'clicks_prev_30d', 'sessions_prev_30d',
]
FEATURES_NUMERIC = list(MODEL_NUMERIC_FEATURES)  # 19
FEATURES_CATEGORICAL = list(MODEL_CATEGORICAL_FEATURES)  # 9
FEATURES = FEATURES_NUMERIC + FEATURES_CATEGORICAL  # 28 total
EXCLUDED = [
    'provider_used', 'model_used', 'char_count_tier',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'engaged_sessions_90d', 'pageviews_90d', 'users_90d', 'scroll_events_90d',
    'age_tier_order', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity',
]

# --- Check 1: every raw columns in exactly one bucket ---
all_buckets = {
    'Context': CONTEXT,
    'Label / proxy': LABEL_PROXY,
    'Features': FEATURES,
    'Excluded': EXCLUDED,
}

all_assigned = []
for bucket_name, cols in all_buckets.items():
    for c in cols:
        all_assigned.append(c)
dupes = [c for c in set(all_assigned) if all_assigned.count(c) > 1]
print('=== BUCKET CHECK 1: mutual exclusivity (expect 0 duplicates)')
print(f'  Duplicate assignments: {dupes if dupes else "(none — good)"}')

# --- Check 2: union coverage (44 raw + 8 prep = 52 columns ---
all_raw_cols = list(df.columns)
all_full_universe = all_raw_cols + prep_added
missing_prep = [c for c in prep_added if c not in all_assigned]
missing_raw = [c for c in all_raw_cols if c not in all_assigned]
extra_assigned = [c for c in all_assigned if c not in all_full_universe]
print(f'\n=== BUCKET CHECK 2: coverage')
print(f'  Raw CSV cols        : {len(all_raw_cols)} (expect 44)')
print(f'  Prep-added cols      : {len(prep_added)} (expect 8)')
print(f'  Total universe       : {len(all_full_universe)} (expect 52)')
print(f'  Total assigned     : {len(all_assigned)} (expect 52)')
print(f'  Raw cols unassigned: {missing_raw if missing_raw else "(none — good)"}')
print(f'  Prep cols unassigned: {missing_prep if missing_prep else "(none — good)"}')
print(f'  Extra (not in universe): {extra_assigned if extra_assigned else "(none — good)"}')

# --- Check 3: feature lists match ml_utils.py exactly ---
print(f'\n=== BUCKET CHECK 3: match ml_utils.MODEL_*_FEATURES')
num_match_numeric = sorted(FEATURES_NUMERIC) == sorted(MODEL_NUMERIC_FEATURES)
num_match_categorical = sorted(FEATURES_CATEGORICAL) == sorted(MODEL_CATEGORICAL_FEATURES)
print(f'  Numeric features match : {num_match_numeric} (19 expected)')
print(f'  Categorical features match: {num_match_categorical} (9 expected)')

# --- Check 4: counts per bucket + summary table ---
print(f'\n=== BUCKET SUMMARY')
summary = pd.DataFrame([
    {'Bucket': name, 'N columns': len(cols), 'Examples': ', '.join(cols[:4]) + ('...' if len(cols) > 4 else '')}
    for name, cols in all_buckets.items()
])
print(summary.to_string(index=False))

=== BUCKET CHECK 1: mutual exclusivity (expect 0 duplicates)
  Duplicate assignments: (none — good)

=== BUCKET CHECK 2: coverage
  Raw CSV cols        : 44 (expect 44)
  Prep-added cols      : 8 (expect 8)
  Total universe       : 52 (expect 52)
  Total assigned     : 52 (expect 52)
  Raw cols unassigned: (none — good)
  Prep cols unassigned: (none — good)
  Extra (not in universe): (none — good)

=== BUCKET CHECK 3: match ml_utils.MODEL_*_FEATURES
  Numeric features match : True (19 expected)
  Categorical features match: True (9 expected)

=== BUCKET SUMMARY
       Bucket  N columns                                                                Examples
      Context          2                                                   content_id, client_id
Label / proxy          9 trend_direction, trend_pct, is_declining_label, impressions_last_30d...
     Features         26                          search_volume, competition, cpc, word_count...
     Excluded         15          provider_u

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.